<a href="https://colab.research.google.com/github/blankqspace/homework_compling_course/blob/main/homework_rnn_Artamonova.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Домашнее задание № 8

## Задание 1 (4 балла)

Обучите 2 модели похожую по архитектуре на модель из ULMFit для задачи классификации текста (датасет - lenta_40k )
В моделях должно быть как минимум два рекуррентных слоя, а финальный вектор для классификации составляться из последнего состояния RNN (так делалось в семинаре), а также AveragePooling и MaxPooling из всех векторов последовательности (конкатенируйте последнее состояния и результаты пулинга). В первой модели используйте обычные слои, а во второй Bidirectional. Рассчитайте по классовую точность/полноту/f-меру для каждой из модели (результаты не должны быть совсем близкие к нулю после обучения на хотя бы нескольких эпохах).

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset

import pandas as pd
import numpy as np
from string import punctuation
from sklearn.model_selection import train_test_split
from collections import Counter
import matplotlib.pyplot as plt
%matplotlib inline

In [5]:
data = pd.read_csv('lenta_40k.csv.zip')

In [6]:
def preprocess(text):
    tokens = text.lower().split()
    tokens = [token.strip(punctuation) for token in tokens]
    return tokens

vocab = Counter()
for text in data.text:
    vocab.update(preprocess(text))
filtered_vocab = set()
for word in vocab:
    if vocab[word] > 30:
        filtered_vocab.add(word)

word2id = {'PAD': 0, 'UNK': 1}

for word in filtered_vocab:
    word2id[word] = len(word2id)
id2word = {i: word for word, i in word2id.items()}

X = []
for text in data.text:
    tokens = preprocess(text)
    ids = [word2id.get(token, 1) for token in tokens]  # 1 = UNK
    X.append(ids)

MAX_LEN = max(len(x) for x in X)
MEAN_LEN = np.median([len(x) for x in X])

def pad_sequence(seq, max_len, pad_value=0):
    if len(seq) > max_len:
        return seq[:max_len]
    return seq + [pad_value] * (max_len - len(seq))

X = np.array([pad_sequence(x, MAX_LEN) for x in X])
X.shape

id2label = {i: label for i, label in enumerate(set(data.topic.values))}
label2id = {l: i for i, l in id2label.items()}
y = np.array([label2id[label] for label in data.topic.values])
len(label2id)

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.05, stratify=y)

# конвертируем в тензоры PyTorch
X_train = torch.LongTensor(X_train)
X_valid = torch.LongTensor(X_valid)
y_train = torch.LongTensor(y_train)
y_valid = torch.LongTensor(y_valid)

# создаем DataLoader для батчей
BATCH_SIZE = 256
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(TensorDataset(X_valid, y_valid), batch_size=BATCH_SIZE, shuffle=False)

In [8]:
# модель 1 обычные слои

class ULMFitLikeClassifier(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_size, output_dim, num_layers=2):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)

        self.rnn = nn.LSTM(emb_dim, hidden_size,num_layers=num_layers, batch_first=True, dropout=0.3)

        self.fc = nn.Linear(hidden_size * 3, output_dim)

    def forward(self, text):
        embedded = self.embedding(text)  # [batch, seq, emb]

        output, (h, c) = self.rnn(embedded)
        # output: [batch, seq, hidden]
        # h: [num_layers, batch, hidden]

        last_hidden = h[-1]                 # [batch, hidden]
        avg_pool = torch.mean(output, 1)   # [batch, hidden]
        max_pool, _ = torch.max(output, 1) # [batch, hidden]

        features = torch.cat([last_hidden, avg_pool, max_pool], dim=1)

        logits = self.fc(features)
        return logits

In [9]:
# модель 2 bidirectional слои true

class BiULMFitLikeClassifier(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_size, output_dim, num_layers=2):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)

        self.rnn = nn.LSTM(emb_dim, hidden_size, num_layers=num_layers, batch_first=True, dropout=0.3,bidirectional=True)

        self.fc = nn.Linear(hidden_size * 2 * 3, output_dim)

    def forward(self, text):
        embedded = self.embedding(text)
        output, (h, c) = self.rnn(embedded)
        last_forward = h[-2]
        last_backward = h[-1]
        last_hidden = torch.cat([last_forward, last_backward], dim=1)
        avg_pool = torch.mean(output, 1)
        max_pool, _ = torch.max(output, 1)
        features = torch.cat([last_hidden, avg_pool, max_pool], dim=1)
        logits = self.fc(features)
        return logits

In [10]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model1 = ULMFitLikeClassifier(
    vocab_size=len(word2id),
    emb_dim=100,
    hidden_size=128,
    output_dim=len(label2id)
).to(device)

model2 = BiULMFitLikeClassifier(
    vocab_size=len(word2id),
    emb_dim=100,
    hidden_size=128,
    output_dim=len(label2id)
).to(device)

criterion = nn.CrossEntropyLoss()

optimizer1 = optim.Adam(model1.parameters(), lr=1e-3)
optimizer2 = optim.Adam(model2.parameters(), lr=1e-3)

In [13]:
from sklearn.metrics import f1_score
from tqdm.notebook import tqdm
import numpy as np

EPOCHS = 10

def train_epoch(model, iterator, optimizer, criterion):
    model.train()

    epoch_loss = []
    epoch_f1 = []

    for texts, labels in tqdm(iterator, desc="Training", leave=False):
        texts = texts.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        predictions = model(texts)
        loss = criterion(predictions, labels)
        loss.backward()
        optimizer.step()

        preds = predictions.argmax(1).cpu().numpy()
        y_true = labels.cpu().numpy()

        epoch_loss.append(loss.item())
        epoch_f1.append(f1_score(y_true, preds, average='micro'))

    return np.mean(epoch_loss), np.mean(epoch_f1)


def evaluate(model, iterator, criterion):
    model.eval()

    epoch_loss = []
    epoch_f1 = []

    with torch.no_grad():
        for texts, labels in tqdm(iterator, desc="Evaluating", leave=False):
            texts = texts.to(device)
            labels = labels.to(device)

            predictions = model(texts)
            loss = criterion(predictions, labels)

            preds = predictions.argmax(1).cpu().numpy()
            y_true = labels.cpu().numpy()

            epoch_loss.append(loss.item())
            epoch_f1.append(f1_score(y_true, preds, average='micro'))

    return np.mean(epoch_loss), np.mean(epoch_f1)

for epoch in range(EPOCHS):
    train_loss, train_f1 = train_epoch(model1, train_loader, optimizer1, criterion)
    val_loss, val_f1 = evaluate(model1, valid_loader, criterion)

    print(f"[Model1] Epoch {epoch+1}: Train F1={train_f1:.4f} | Val F1={val_f1:.4f}")

Training:   0%|          | 0/165 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/9 [00:00<?, ?it/s]

[Model1] Epoch 1: Train F1=0.7411 | Val F1=0.6997


Training:   0%|          | 0/165 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/9 [00:00<?, ?it/s]

...

Evaluating:   0%|          | 0/9 [00:00<?, ?it/s]

[Model1] Epoch 9: Train F1=0.9216 | Val F1=0.7199


Training:   0%|          | 0/165 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/9 [00:00<?, ?it/s]

[Model1] Epoch 10: Train F1=0.9318 | Val F1=0.7208


In [15]:
for epoch in range(EPOCHS):
    train_loss, train_f1 = train_epoch(model2, train_loader, optimizer2, criterion)
    val_loss, val_f1 = evaluate(model2, valid_loader, criterion)

    print(f"[Model2] Epoch {epoch+1}: Train F1={train_f1:.4f} | Val F1={val_f1:.4f}")

Training:   0%|          | 0/165 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/9 [00:00<?, ?it/s]

[Model2] Epoch 1: Train F1=0.8370 | Val F1=0.7393


Training:   0%|          | 0/165 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/9 [00:00<?, ?it/s]

...

Evaluating:   0%|          | 0/9 [00:00<?, ?it/s]

[Model2] Epoch 9: Train F1=0.9893 | Val F1=0.7438


Training:   0%|          | 0/165 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/9 [00:00<?, ?it/s]

[Model2] Epoch 10: Train F1=0.9940 | Val F1=0.7410


In [22]:
from sklearn.metrics import classification_report

def get_classification_report(model, loader):
    model.eval()
    all_preds = []
    all_true = []

    with torch.no_grad():
        for texts, labels in loader:
            texts = texts.to(device)
            preds = model(texts).argmax(1).cpu().numpy()

            all_preds.extend(preds)
            all_true.extend(labels.numpy())

    print(classification_report(all_true, all_preds))

In [23]:
print("Model 1")
get_classification_report(model1, valid_loader)

print("Model 2")
get_classification_report(model2, valid_loader)

Model 1


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

           0       0.77      0.77      0.77       239
           1       0.75      0.82      0.78       159
           2       0.71      0.52      0.60        23
           3       0.38      0.44      0.41        84
           5       0.48      0.37      0.42        60
           6       0.38      0.23      0.29        22
           7       0.00      0.00      0.00         1
           8       0.76      0.67      0.71        66
           9       0.78      0.81      0.80       159
          10       0.76      0.69      0.72       160
          11       0.00      0.00      0.00         4
          12       0.74      0.73      0.73       410
          13       0.94      0.92      0.93       195
          14       0.00      0.00      0.00         2
          15       0.56      0.52      0.54       132
          16       0.50      0.33      0.40        21
          18       0.70      0.76      0.73       481

    accuracy              

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## Задание 2 (6 баллов)


На данных википедии (wikiann) обучите и сравните 3 модели:  
1) модель в которой как минимум два рекуррентных слоя, причем один из них GRU, а другой LSTM
2) модель в которой как минимум 3 рекуррентных слоя идут друг за другом и при этом 2-ой и 3-й слои еще имеют residual connection к изначальным эмбедингам. Для того, чтобы сделать residual connection вам нужно будет использовать одинаковую размерность эмбедингов и количество unit'ов в RNN слоях, чтобы их можно было просуммировать
3) модель в которой будут и рекуррентные и сверточные слои (как минимум 2 rnn и как минимум 2 cnn слоя). В cnn слоях будьте аккуратны с укорачиванием последовательности и используйте паддинг



Сравните качество по метрикам (точность/полнота/f-мера). Также придумайте несколько сложных примеров и проверьте, какие сущности определяет каждая из моделей.

In [5]:
from datasets import load_dataset
dataset = load_dataset("unimelb-nlp/wikiann", 'ru')

vocab = Counter()
for sent in dataset['train']['tokens']:
    vocab.update([x.lower() for x in sent])

word2id = {'PAD': 0, 'UNK': 1}
for word in vocab:
    word2id[word] = len(word2id)

def encode_sentences(sentences, word2id):
    return [[word2id.get(w.lower(), 1) for w in sent] for sent in sentences]

def pad_sequences(sequences, max_len, pad_value=0):
    return np.array([seq[:max_len] + [pad_value] * max(0, max_len - len(seq)) for seq in sequences])

X_train = encode_sentences(dataset['train']['tokens'], word2id)
X_test = encode_sentences(dataset['test']['tokens'], word2id)

MAX_LEN = max(len(x) for x in X_train)
X_train = pad_sequences(X_train, MAX_LEN)
X_test = pad_sequences(X_test, MAX_LEN)

id2label = {0: "O", 1: "B-PER", 2: "I-PER", 3: "B-ORG", 4: "I-ORG", 5: "B-LOC", 6: "I-LOC", 7: "PAD"}
label2id = {v: k for k, v in id2label.items()}

# PAD индекс = 7 для ignore_index в CrossEntropyLoss
y_train = pad_sequences(dataset['train']['ner_tags'], MAX_LEN, pad_value=7)
y_test = pad_sequences(dataset['test']['ner_tags'], MAX_LEN, pad_value=7)

X_train_t = torch.LongTensor(X_train)
X_test_t = torch.LongTensor(X_test)
y_train_t = torch.LongTensor(y_train)
y_test_t = torch.LongTensor(y_test)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=128, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=128, shuffle=False)

In [6]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0

    for x, y in tqdm(loader, leave=False):
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        logits = model(x)  # [batch, seq, classes]
        loss = criterion(logits.view(-1, logits.shape[-1]), y.view(-1))
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


def evaluate(model, loader):
    model.eval()
    preds, true = [], []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            logits = model(x)
            p = logits.argmax(-1).cpu().numpy()
            t = y.numpy()

            mask = t != 7
            preds.extend(p[mask])
            true.extend(t[mask])

    print(classification_report(true, preds, target_names=list(label2id.keys())[:-1]))

In [7]:
# модель 1
class GRU_LSTM_Model(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.gru = nn.GRU(emb_dim, hidden, batch_first=True, bidirectional=True)
        self.lstm = nn.LSTM(hidden*2, hidden, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden*2, num_classes)

    def forward(self, x):
        x = self.embedding(x)
        x, _ = self.gru(x)
        x, _ = self.lstm(x)
        return self.fc(x)

In [8]:
# модель 2

class ResidualRNN(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.rnn1 = nn.LSTM(emb_dim, hidden, batch_first=True, bidirectional=True)
        self.rnn2 = nn.LSTM(hidden*2, hidden, batch_first=True, bidirectional=True)
        self.rnn3 = nn.LSTM(hidden*2, hidden, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden*2, num_classes)

    def forward(self, x):
        emb = self.embedding(x)

        x, _ = self.rnn1(emb)
        x, _ = self.rnn2(x + torch.cat([emb, emb], dim=2))
        x, _ = self.rnn3(x + torch.cat([emb, emb], dim=2))

        return self.fc(x)

In [9]:
# модель 3

class RNN_CNN_Model(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)

        self.rnn1 = nn.LSTM(emb_dim, hidden, batch_first=True, bidirectional=True)
        self.rnn2 = nn.LSTM(hidden*2, hidden, batch_first=True, bidirectional=True)

        self.conv1 = nn.Conv1d(hidden*2, hidden*2, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(hidden*2, hidden*2, kernel_size=3, padding=1)

        self.fc = nn.Linear(hidden*2, num_classes)

    def forward(self, x):
        x = self.embedding(x)
        x, _ = self.rnn1(x)
        x, _ = self.rnn2(x)

        x = x.transpose(1, 2)
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = x.transpose(1, 2)

        return self.fc(x)

In [10]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
criterion = nn.CrossEntropyLoss(ignore_index=7)

In [14]:
from sklearn.metrics import classification_report
from tqdm.notebook import tqdm
import numpy as np

model11 = GRU_LSTM_Model(len(word2id), 128, 128, len(label2id)).to(device)
optimizer11 = torch.optim.Adam(model11.parameters(), lr=1e-3)

for epoch in range(5):
    loss = train_epoch(model11, train_loader, optimizer11, criterion)
    print(f"[GRU+LSTM] Epoch {epoch+1}, loss={loss:.4f}")

evaluate(model11, test_loader)

  0%|          | 0/157 [00:00<?, ?it/s]

[GRU+LSTM] Epoch 1, loss=0.8805


  0%|          | 0/157 [00:00<?, ?it/s]

[GRU+LSTM] Epoch 2, loss=0.4835


  0%|          | 0/157 [00:00<?, ?it/s]

[GRU+LSTM] Epoch 3, loss=0.3527


  0%|          | 0/157 [00:00<?, ?it/s]

[GRU+LSTM] Epoch 4, loss=0.2541


  0%|          | 0/157 [00:00<?, ?it/s]

[GRU+LSTM] Epoch 5, loss=0.1711
              precision    recall  f1-score   support

           O       0.93      0.94      0.94     40480
       B-PER       0.93      0.78      0.85      3543
       I-PER       0.92      0.89      0.91      7544
       B-ORG       0.55      0.74      0.63      4074
       I-ORG       0.75      0.82      0.78      8008
       B-LOC       0.87      0.64      0.74      4559
       I-LOC       0.89      0.68      0.77      3060

    accuracy                           0.87     71268
   macro avg       0.84      0.79      0.80     71268
weighted avg       0.88      0.87      0.88     71268



In [15]:
model22 = ResidualRNN(len(word2id), 128, 128, len(label2id)).to(device)
optimizer22 = torch.optim.Adam(model22.parameters(), lr=1e-3)

for epoch in range(5):
    loss = train_epoch(model22, train_loader, optimizer22, criterion)
    print(f"[Residual] Epoch {epoch+1}, loss={loss:.4f}")

evaluate(model22, test_loader)

  0%|          | 0/157 [00:00<?, ?it/s]

[Residual] Epoch 1, loss=0.8354


  0%|          | 0/157 [00:00<?, ?it/s]

[Residual] Epoch 2, loss=0.4560


  0%|          | 0/157 [00:00<?, ?it/s]

[Residual] Epoch 3, loss=0.3083


  0%|          | 0/157 [00:00<?, ?it/s]

[Residual] Epoch 4, loss=0.1925


  0%|          | 0/157 [00:00<?, ?it/s]

[Residual] Epoch 5, loss=0.1114
              precision    recall  f1-score   support

           O       0.93      0.95      0.94     40480
       B-PER       0.96      0.72      0.82      3543
       I-PER       0.96      0.80      0.87      7544
       B-ORG       0.59      0.72      0.65      4074
       I-ORG       0.70      0.83      0.76      8008
       B-LOC       0.82      0.72      0.77      4559
       I-LOC       0.81      0.77      0.78      3060

    accuracy                           0.87     71268
   macro avg       0.83      0.78      0.80     71268
weighted avg       0.88      0.87      0.87     71268



In [16]:
model3 = RNN_CNN_Model(len(word2id), 128, 128, len(label2id)).to(device)
optimizer3 = torch.optim.Adam(model3.parameters(), lr=1e-3)

for epoch in range(5):
    loss = train_epoch(model3, train_loader, optimizer3, criterion)
    print(f"[RNN+CNN] Epoch {epoch+1}, loss={loss:.4f}")

evaluate(model3, test_loader)

  0%|          | 0/157 [00:00<?, ?it/s]

[RNN+CNN] Epoch 1, loss=0.9185


  0%|          | 0/157 [00:00<?, ?it/s]

[RNN+CNN] Epoch 2, loss=0.4757


  0%|          | 0/157 [00:00<?, ?it/s]

[RNN+CNN] Epoch 3, loss=0.3298


  0%|          | 0/157 [00:00<?, ?it/s]

[RNN+CNN] Epoch 4, loss=0.2232


  0%|          | 0/157 [00:00<?, ?it/s]

[RNN+CNN] Epoch 5, loss=0.1407
              precision    recall  f1-score   support

           O       0.90      0.96      0.93     40480
       B-PER       0.89      0.79      0.84      3543
       I-PER       0.94      0.86      0.90      7544
       B-ORG       0.75      0.55      0.63      4074
       I-ORG       0.84      0.69      0.75      8008
       B-LOC       0.69      0.66      0.67      4559
       I-LOC       0.65      0.79      0.71      3060

    accuracy                           0.86     71268
   macro avg       0.81      0.76      0.78     71268
weighted avg       0.86      0.86      0.86     71268



In [56]:
print("MAX TRUE LABEL:", y_test_t.max().item())

MAX TRUE LABEL: 7


In [22]:

def predict_sentence(model, sentence):
    tokens = sentence.split()
    ids = [word2id.get(w.lower(), 1) for w in tokens]
    x = torch.LongTensor([ids]).to(device)

    with torch.no_grad():
        preds = model(x).argmax(-1)[0].cpu().numpy()

    return list(zip(tokens, [id2label[p] for p in preds]))

sentences = [
    "Арбитражный суд города федерального значения Москвы удовлетворил ходатайство Генеральной прокуратуры Российской Федерации о наложении ареста на активы транснационального холдинга «Сибирский горно-металлургический альянс» (СГМА) в рамках дела о нарушении антимонопольного законодательства Евразийского экономического союза (ЕАЭС).",
    "Руководитель Федеральной службы по надзору в сфере защиты прав потребителей и благополучия человека (Роспотребнадзор) доложила на заседании президиума Государственного совета Российской Федерации о реализации национального проекта «Здравоохранение» в части создания опорной сети центров общественного здоровья." ]

for s in sentences:
    print(s)
    print("Model1:", predict_sentence(model11, s))
    print("Model2:", predict_sentence(model22, s))
    print("Model3:", predict_sentence(model3, s))
    print()

Арбитражный суд города федерального значения Москвы удовлетворил ходатайство Генеральной прокуратуры Российской Федерации о наложении ареста на активы транснационального холдинга «Сибирский горно-металлургический альянс» (СГМА) в рамках дела о нарушении антимонопольного законодательства Евразийского экономического союза (ЕАЭС).
Model1: [('Арбитражный', 'B-ORG'), ('суд', 'I-ORG'), ('города', 'I-ORG'), ('федерального', 'I-ORG'), ('значения', 'I-ORG'), ('Москвы', 'I-ORG'), ('удовлетворил', 'I-ORG'), ('ходатайство', 'I-ORG'), ('Генеральной', 'I-ORG'), ('прокуратуры', 'I-ORG'), ('Российской', 'I-ORG'), ('Федерации', 'I-ORG'), ('о', 'I-ORG'), ('наложении', 'I-ORG'), ('ареста', 'I-ORG'), ('на', 'O'), ('активы', 'B-ORG'), ('транснационального', 'I-ORG'), ('холдинга', 'I-ORG'), ('«Сибирский', 'I-ORG'), ('горно-металлургический', 'I-ORG'), ('альянс»', 'I-ORG'), ('(СГМА)', 'I-ORG'), ('в', 'O'), ('рамках', 'O'), ('дела', 'I-ORG'), ('о', 'O'), ('нарушении', 'I-ORG'), ('антимонопольного', 'I-ORG'), 